# Rivulet: U. S. Geological Survey Water Quality eXchange
_by Michelle H Wilkerson_

### Purpose of this Notebook

This notebook focuses on **Water Quality** as a phenomenon. While most students understand that poor water quality can impact health, they may not know what sorts of pollutants are involved, or what kinds of events or conditions lead to reductions in water quality.

This data tool allows users to connect to the United States Geological Survey (USGS) water data APIs, search for water quality data streams in an area of interest, and then provides a collection of ways to filter and search the data to ensure you find datasets that have patterns worth exploring. 

<details>
    <summary>Click here for more information about APIs</summary>

API stands for **Application Programming Interface**. In data science, APIs are incredibly common because they allow different computer programs to access to "fetch", or download, datasets from large databases or services. Instead of downloading a massive CSV file that might be out of date by the time you open it, an API lets you request exactly the data you need, right when you need it. 

For educators, APIs are powerful because they allow students to work with **real-world, live data**. However, they do come with some risks: sometimes an API service might be down, or the way you ask for data (the "request") might need to change if the service is updated. Learning to use APIs is a core skill in modern science, helping students understand that data isn't just a static table, but a dynamic resource that we can query and explore.

You are welcome to modify and adapt this script. You may find the USGS' water data APIs documentation [here]() and [here](https://doi-usgs.github.io/dataretrieval-python/) helpful.

This notebook was developed as part of NSF Grant 2445609 to support accessing and processing public data for middle and high school classroom activities. It's written to be relatively accessible to beginners, but if you have not interacted with computational notebooks or python before you may find navigating this tool difficult. (Check out the [Show Your Work](https://github.com/CalCoRE/show-your-work) project for a gentle introduction to computational notebooks for educators!)

Our project is focused on supporting data analysis and mechanistic reasoning in science education. In other words, we want students to learn how data provides information about _how scientific mechanisms work_, and how understanding scientific mechanisms can help them to _explain and interpret patterns in data_. This builds on a long history of research on complex systems and agent-based modeling, and more closely connects that work to current expansions of data analysis across subjects.

# Part I: Setup

The USGS has developed a Python library (unhelpfully but impressively called `dataretrieval`) to help people access and fetch hydrological data from several different water-related data services.


### Connect to NWIS

You can sign up for an API key [at this site](https://api.waterdata.usgs.gov/signup). Once you receive it, replace the DEMO_KEY below with your unique API key. This will give you more reliable access to the data, and will help USGS understand more about who is using their services. Do not share your key!

In [ ]:
%pip install dataretrieval

API_KEY = "DEMO_KEY" 

Not sure this is important yet, but the docs say that if you want data after March 2024 you want to specify `legacy=False`. See [here](https://github.com/DOI-USGS/dataretrieval-python#:~:text=%E2%9A%A0%EF%B8%8F,the%20wqp%20module.) and [here](https://doi-usgs.github.io/dataRetrieval/articles/Status.html#:~:text=Discrete%20Data,non%2DUSGS%20data) for more information.

In [ ]:
import dataretrieval.nwis as nwis
import pandas as pd

### Specify a Location

Let's create a bounding box to indicate the region we are interested in. We will then filter our queries to focus only on monitoring sites within the bounding box.

In [ ]:
# EDIT HERE: Define a bounding box around your
# target region. If it is densely populated, we suggest
# you start with a bounding box that is only one degree
# in area. 

min_lat = 37.5 # CHANGE TO YOUR MINIMUM LATITUDE
max_lat = 38 # CHANGE TO YOUR MAXIMUM LATITUDE

min_long = -122.5 # CHANGE TO YOUR MINIMUM LONGITUDE
max_long = -122

# this is unnecessary but sort of luxurious. let's map the box to
# make sure we're capturing what we want.

import folium

bbox = [[min_lat, min_long], [max_lat, max_long]]

# Calculate the center of the box to position the map
map_center = [(bbox[0][0] + bbox[1][0]) / 2, (bbox[0][1] + bbox[1][1]) / 2]

# Create a Folium map object
m = folium.Map(location=map_center, zoom_start=8)

# Add a rectangle for the bounding box to the map
folium.Rectangle(
    bounds=bbox,
    color="#ff0000",        # Red border
    fill=True,
    fill_color="#ff7800",   # Orange fill
    fill_opacity=0.2
).add_to(m)

m

The function what_sites() allows us to see which sites satisfy certain criteria, like falling within a specific bounding box (how we use it here) or measuring a particular thing (which we will do later). The code below takes the latitudes and longitudes you entered above, and passes them to NWIS to see what data sites they have within that area.

In [ ]:
bbox = str(min_long) + "," + str(min_lat) + "," + str(max_long) + "," + str(max_lat)

sites, sites_metadata = nwis.what_sites(bBox=bbox)
sites

Whoa! You are likely looking at a long list of sites. Depending on the phenomena we're interested in, we'll filter this to only what we need. Or, you can use the list above to help you identify a specific site if you know of one to use below.

### Identify a Target Date

Next, we want to identify a focus date. This might be a time when we think something interesting might have happened, or a date we're interested in for other reasons. California experienced king tides during Jan 11-13 in 2026, so we'll identify Jan 12 as a target date. You can edit it to whatever date you want.

In [ ]:
# EDIT HERE: identify a target date when something interesting
# was happening. Below, I define Jan 12, 2025, during king tides in CA.

from datetime import datetime, timedelta

target_date = "01-12-2026"
target_datetime = datetime.strptime(target_date, "%m-%d-%Y")

start_date = target_datetime
end_date = target_datetime + timedelta(days=1)

Now that we've identified a general location and time period, let's dive into the data.

# Part II: Fetch Data

## Salinity in Estuaries (Exploring Tides)

This first section helps you select datasets that feature information about **salinity** (saltiness) in bodies of water. There are lots of places where salinity behaves in interesting ways: in estuaries where fresh and salt water meet and move in and out with the tides, and in cold areas during winter months, when salt road treatments run off into freshwater sources. 

Salinity is measured through electrical conductance. In NWIS, there are many salinity measures, but the best one to look for here is "specific conductance", which is a measure of salinity that is also adjusted for temperature and that is used by both the USGS and the EPA. You can choose other measures, see the full list in [USGS parameter codes](https://help.waterdata.usgs.gov/parameter_cd?group_cd=PHY). 

Below, we'll look for all the sites within the region you specified that are identified as estuaries and that are measured for specific conductance. A lot of these don't actually have the data we need, so I'm applying an extra filter to make sure we get back only sites with actual measurements.

In [ ]:
estuaries, estuaries_metadata = nwis.what_sites(bBox=bbox,
                                        startDt=start_date,
                                        endDt=end_date,
                                        parameterCd='00095',
                                        siteType='ES') #estuaries

estuaries = estuaries[~estuaries['alt_datum_cd'].isna()] #remove if NaN

estuaries

To really see how salinity in an estuary changes as the tides flow in and out, you will need a week or two of data around the target date. 

Pick your favorite site from the list above, and plop it into the code below. When you run the code, it will fetch the specific conductance data for two weeks around your target date.

In [ ]:
start_date = target_datetime - timedelta(days=7)
end_date = target_datetime + timedelta(days=7)

sal_data, meta = nwis.get_iv(sites='373015122071000', #site of your choice 
                           parameterCd='00095',
                           start=start_date.strftime("%Y-%m-%d"),
                           end=end_date.strftime("%Y-%m-%d"))

sal_data

NWIS reports this data at different depths, at 4 feet from the sea floor at that location (the "bed") and at 25 feet from the floor. Salt water is heavier, so it's expected that there will be a higher conductance deeper in the water. Comparing these measures can also reveal information about whether the water has been mixing a lot in this location.

Now, let's plot the salinity of water in the shallower part of this site. Will we see the impact of the tides?

In [ ]:
import seaborn as sns

sns.lineplot(data=sal_data,x='datetime', y='00095_upper: 25 ft from bed')

You should see a daily rise and fall of salinity, representing when the tide pushes ocean water into the estuary and when the outflow of fresh water flushes it back out as the tides recede. 

If this data look interesting and you'd like to use them with another program for deeper analysis, use the code below to save it as a CSV file.

In [ ]:
sal_data.to_csv("salinity_data.csv")

## EColi

Ecoli problems occur in water sources after rains, when sewers release bacteria. Let's check it out. Remember that you defined your location bounding box and your target date in Part 2 setup above. Go up and reset these if you have been working through the notebook in order, to identify a time close to a rain, perhaps after a dry spell.

For SF Bay area, we're going to look for creeks. We're going to use the target date of Sept 9 because I remember that's when I got all wet biking to campus :)

In [ ]:
### eventually clean this up so target date is taken care of in 
# the setup section only but for now I'm just going
# to do it here
target_date = "01-04-2023"
target_datetime = datetime.strptime(target_date, "%m-%d-%Y")

start_date = target_datetime
end_date = target_datetime + timedelta(days=1)
## end cleanup section

# check the parameter here, I went for one that was there and made sense
ecoli_data, ec_meta = nwis.what_sites(bBox=bbox,
                                      startDt=start_date,
                                      endDt=end_date,
                                      parameterCd='31625')#, #fecal coliforms
                                    #  siteType='ST') #streams

ecoli_data

Ok just FYI you can search for body parts as an attribute that's available for what's in the water.

Note that for e coli data, we use a different module from the one we used in section 3 estuaries. Here, we use a module called WQP. More info [here]('https://code.usgs.gov/water/dataretrieval-python/-/blob/v1.0.2/dataretrieval/wqp.py')

In [ ]:
#about a week surrounding target date
start_date = target_datetime - timedelta(days=3)
end_date = target_datetime + timedelta(days=3)

from dataretrieval import wqp

ec_data, ecmeta = wqp.get_results(siteid='11181330', #site of your choice 
                           pCode='31625,50468,90901',
                           startDateLo=start_date.strftime("%m-%d-%Y"),
                           startDateHi=end_date.strftime("%m-%d-%Y"))

ec_data

#50468	E. coli, modified m-TEC method	Most Probable Number per 100 mL (MPN/100 mL)
#90901	E. coli, unspecified method	Colonies per 100 mL (cfu/100 mL)
#31633	E. coli, m-TEC MF method	Colony Forming Units per 100 mL (cfu/100 mL)
#31648	E. coli, m-TEC MF method	Colonies per 100 mL (cfu/100 mL)

E Coli is most likely to be found the sites most likely to show dramatic increases in E. coli after a rain event are smaller, fast-reacting bodies of water in watersheds with significant sources of fecal contamination. These are often called "flashy" systems.

in order of likelihood of a dramatic effect, check out sites that are:
- storm drains
- urban creeks and streams
- coastal beaches and river mouths
- small lakes

## Oxygen Dissolution

Dissolved Oxygen (DO) is a vital sign for aquatic life. Just like we need oxygen in the air to breathe, fish and other underwater creatures need oxygen that is dissolved in the water. 

How does oxygen get there? It mostly comes from two places: the atmosphere (especially where water is turbulent, like rapids or waterfalls) and as a byproduct of photosynthesis from aquatic plants. 

However, oxygen can also be used up. Microbes that decompose organic matter (like dead algae or sewage) consume oxygen. If too much organic matter enters the water, the oxygen levels can drop so low that fish cannot survive—this is called **hypoxia**. 

Let's look for DO data using the parameter code `00300`.

In [ ]:
# Find sites with Dissolved Oxygen data
do_sites, do_meta = nwis.what_sites(bBox=bbox,
                                   startDt=start_date,
                                   endDt=end_date,
                                   parameterCd='00300')

do_sites.head()

In [ ]:
# Pick a site and fetch the data
do_data, do_meta = nwis.get_iv(sites='11181330', # Example site
                             parameterCd='00300',
                             start=start_date.strftime("%Y-%m-%d"),
                             end=end_date.strftime("%Y-%m-%d"))

if not do_data.empty:
    sns.lineplot(data=do_data, x='datetime', y='00300_Inst')
else:
    print("No data found for this site and period.")

## Nitrates

Nitrates are a bit of a "double-edged sword." They are essential nutrients for plant growth, which is why they are found in most fertilizers. However, when too many nitrates wash into rivers and lakes, they trigger **eutrophication**. 

Here is the mechanism: 
1. Excess nitrates cause a massive bloom of algae.
2. Eventually, the algae die and sink to the bottom.
3. Bacteria move in to decompose the dead algae, using up all the dissolved oxygen in the process.
4. The result? A "dead zone" where other aquatic life can't breathe.

Nitrates can also be dangerous in drinking water, especially for infants. Let's explore Nitrate data (parameter code `00630`).

In [ ]:
# Search for Nitrates using the WQP module
nitrate_data, n_meta = wqp.get_results(bBox=bbox,
                                      pCode='00630',
                                      startDateLo=start_date.strftime("%m-%d-%Y"),
                                      startDateHi=end_date.strftime("%m-%d-%Y"))

nitrate_data.head()

## Lead

Lead is a heavy metal that is highly toxic to humans, especially children. Unlike other pollutants, it rarely occurs naturally in our water sources. Instead, it enters the water through the corrosion of pipes, solder, and fixtures. 

The mechanism here is chemical: **plumbosolvency**. This describes how likely lead is to dissolve into water. Factors like low pH (acidic water) or low mineral content (soft water) make the water more corrosive, which increases the amount of lead that leaches from old plumbing into our drinking water. 

Let's look for Lead measurements using parameter code `01051`.

In [ ]:
# Search for Lead using the WQP module
lead_data, l_meta = wqp.get_results(bBox=bbox,
                                   pCode='01051',
                                   startDateLo=start_date.strftime("%m-%d-%Y"),
                                   startDateHi=end_date.strftime("%m-%d-%Y"))

lead_data.head()

## Credits

We are grateful to Dr. Lex Van Geen for his insights into pedagogically productive datasets.

Hodson, T.O., Hariharan, J.A., Black, S., and Horsburgh, J.S., 2023, dataretrieval (Python): a Python package for discovering and retrieving water data available from U.S. federal hydrologic web services: U.S. Geological Survey software release, https://doi.org/10.5066/P94I5TX3.